# Increasing the order and conditioning

In [ ]:
#    APM41012EP course notebook - Chapter 3 - M. Massot 2026-2027 - École polytechnique
#    ----------   
#    Increasing the order: accuracy, conditioning and stability 
#    
#    Authors: L. Séries and M. Massot - (C) 2026

In [ ]:
import numpy as np
from sympy import Rational, Poly, symbols
import plotly.graph_objs as go
from scipy.integrate import newton_cotes
from scipy.special import roots_legendre
import warnings
warnings.filterwarnings('ignore')

In this part, we are going to focus on the integration of the simple function $f(x) = \cos(2x)$ on the interval $[0,1]$, whose exact result is $0.5 \sin(2)$. The aim here is to understand the impact of increasing the order on the accuracy of the methods (Newton-Cotes, Clenshaw-Curtis and Gauss-Legendre) and to identify, potentially, the conditioning of the problem and the stability of the proposed algorithms.

In [ ]:
def f(x):
    return np.cos(2*x)

res_exa = 0.5*np.sin(2)

## Newton-Cotes formulas  

### Weights computed with scipy 

In [ ]:
s = np.arange(2, 41, 1)

err = np.zeros(s.size)

for i, si in enumerate(s):
    b, _ = newton_cotes(si-1, equal=1)
    b = b/(si-1)
    c = np.linspace(0, 1, si)
    res = np.sum(b * f(c))
    err[i] = np.abs(res - res_exa)
    
fig = go.Figure()
fig.add_trace(go.Scatter(x=s, y=err, mode='lines+markers'))
fig.update_xaxes(title="Number of quadrature points")
fig.update_yaxes(type="log", exponentformat = 'e', title="Error")
fig.show()

This graph clearly shows a difficulty when the order is increased. The impact is so strong in this range of numbers of quadrature points, in connection with the interpolation studies carried out in the previous chapter, that it must be the combination of poor conditioning and of an unstable algorithm. To check this, we take another angle of attack on the computation of the weights by using symbolic computation (one could also use Sagemath with an exact computation in the field of rationals) in order to guarantee that the evaluation of the weights is done accurately.

### Weights computed with sympy (exact rational arithmetic)

In [ ]:
def newton_cotes_closed_exact(n):
    """Closed Newton-Cotes weights of the n+1 equispaced points of [0,1],
       computed exactly in rational arithmetic, then rounded to float64."""
    t = symbols('t')
    c = [Rational(i, n) for i in range(n+1)]
    w = []
    for i in range(n+1):
        lag = Poly(1, t)
        for j in range(n+1):
            if (j!=i): lag = lag * Poly((t - c[j])/(c[i] - c[j]), t)
        prim = lag.integrate()
        w.append(prim.eval(1) - prim.eval(0))
    return np.array([float(wi) for wi in w])

s = np.array([2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 16, 18, 20, 24, 28, 32, 36, 40])

err = np.zeros(s.size)
res_nc_exact = np.zeros(s.size)
b_nc_exact = []

for i, si in enumerate(s):
    b_nc_exact.append(newton_cotes_closed_exact(si-1))
    c = np.linspace(0, 1, si)
    res_nc_exact[i] = np.sum(b_nc_exact[i] * f(c))
    err[i] = np.abs(res_nc_exact[i] - res_exa)

fig = go.Figure()
fig.add_trace(go.Scatter(x=s, y=err, mode='lines+markers'))
fig.update_xaxes(title="Number of quadrature points")
fig.update_yaxes(type="log", exponentformat = 'e', title="Error")
fig.show()

The reader can observe that the degradation of the accuracy is also present, certainly due to a conditioning problem (equidistributed points), but much weaker. One must therefore be careful and assess the conditioning of the problem. It is not difficult to make the connection with the theoretical conditioning studied in the previous chapter, but one can also propose a numerical estimate of it in a perturbative form.

### Perturbation of the $f(c_i)$

In [ ]:
eps = 0.01

In [ ]:
err = np.zeros(s.size)
err_pert = np.zeros(s.size)

for i, si in enumerate(s):
    c = np.linspace(0, 1, si)
    f_pert = f(c) + eps*(2*np.random.rand(c.size)-1)*f(c)
    res_pert = np.sum(b_nc_exact[i] * f_pert)
    err_pert[i] = np.abs(res_nc_exact[i] - res_pert)
    
fig = go.Figure()
fig.add_trace(go.Scatter(x=s, y=err_pert, mode='lines+markers'))
fig.update_xaxes(title="Number of quadrature points")
fig.update_yaxes(type="log", exponentformat = 'e', title="|Res. - Perturbed res.|")
fig.show()

We recover here (the perturbation is random, so try several times...) a conditioning of the order of $10.e7$ for 40 stages, showing that the loss of accuracy observed on the "forward" error is indeed related to a conditioning difficulty.

## Clenshaw-Curtis formula

In [ ]:
def xcheb(n):
    if n == 0:
        return np.array([])
    else:
        x = 0.5*np.cos( (2*np.arange(0,n)+1)*np.pi / (2*n) ) + 0.5
    return x

def coeffs_clenshawcurtis(N):
    x = xcheb(N+1) # n+1 Chebyshev nodes
    tab_k = np.arange(0, N+1, dtype='float')
    b = np.zeros(N+1)
    b[0::2] = 2 / (1 - tab_k[0::2]*tab_k[0::2])
    theta = (2*np.arange(0,N+1)+1)/(2*(N+1))*np.pi 
    ## Construction of the matrix B from the notes
    B = np.cos(np.outer(np.arange(0,N+1),theta))/(N+1)
    B[1:,:] = 2*B[1:,:]
    w = np.dot(np.transpose(B), b)
    return x, 0.5*w    

def coeffs_clenshawcurtis_old(n):
    x = xcheb(n+1) # n+1 Chebyshev nodes
    tab_k = np.arange(0, n+1, dtype='float')
    b = 1 / (tab_k + 1)
    M = np.vander(x, increasing=True)
    w = np.linalg.solve(np.transpose(M), b)
    return x, w    

In [ ]:
s = np.arange(2, 41, 1)

err = np.zeros(s.size)

for i, si in enumerate(s):
    c, b = coeffs_clenshawcurtis(si-1)
    res = np.sum(b * f(c))
    err[i] = np.abs(res - res_exa)
    if (err[i]==0): err[i]=1e-16    

fig = go.Figure()
fig.add_trace(go.Scatter(x=s, y=err, mode='lines+markers'))
fig.update_xaxes(title="Number of quadrature points")
fig.update_yaxes(type="log", exponentformat = 'e', title="Error")
fig.show()

Even though a more accurate estimate of the integration weights could lead to a complete absence of loss of accuracy, we see that the conditioning is more favorable here, as could be expected following Chapter 2. 

### Perturbation of the $f(c_i)$

In [ ]:
eps = 0.01

In [ ]:
err_pert = np.zeros(s.size)

for i, si in enumerate(s):
    c, b = coeffs_clenshawcurtis(si-1)
    res = np.sum(b * f(c))
    err[i] = np.abs(res - res_exa)
    f_pert = f(c) + eps*(2*np.random.rand(c.size)-1)*f(c)
    res_pert = np.sum(b * f_pert)
    err_pert[i] = np.abs(res - res_pert)

fig = go.Figure()
fig.add_trace(go.Scatter(x=s, y=err_pert, mode='lines+markers'))
fig.update_xaxes(title="Number of quadrature points")
fig.update_yaxes(type="log", exponentformat = 'e', title="|Res. - Perturbed res.|")
fig.show()

We see here that the conditioning remains very reasonable, but the attentive reader will notice that in order to estimate the conditioning of a problem numerically, one has to guarantee that the numerical algorithm has excellent stability, otherwise the conclusions are tarnished. A more accurate estimate of the weights (symbolic computation) would make it possible here to highlight the excellent conditioning of the problem.

**Result for the function $g(x) = \sqrt(x) \log(x)$**

In [ ]:
def g(x):
    return  np.sqrt(x)*np.log(x)

res_exa_g = -4./9.

In [ ]:
s = np.arange(2, 41, 1)

err = np.zeros(s.size)

for i, si in enumerate(s):
    c, b = coeffs_clenshawcurtis(si-1)
    res = np.sum(b * g(c))
    err[i] = np.abs(res - res_exa_g)

fig = go.Figure()
fig.add_trace(go.Scatter(x=s, y=err, mode='lines+markers'))
fig.update_xaxes(title="Number of quadrature points")
fig.update_yaxes(type="log", exponentformat = 'e', title="Error")
fig.show()

## Gauss formula

In [ ]:
s = np.arange(2, 41, 1)

err = np.zeros(s.size)

for i, si in enumerate(s):
    c, b = roots_legendre(si)
    c = 0.5*(c+1)
    b = 0.5*b
    res = np.sum(b * f(c))
    err[i] = np.abs(res - res_exa)
    if (err[i] == 0): err[i]=1e-16
    
fig = go.Figure()
fig.add_trace(go.Scatter(x=s, y=err, mode='lines+markers'))
fig.update_xaxes(title="Number of quadrature points")
fig.update_yaxes(type="log", exponentformat = 'e', title="Error")
fig.show()

### Perturbed formula 

In [ ]:
eps = 0.01

In [ ]:
s = np.arange(2, 41, 1)

err = np.zeros(s.size)
err_pert = np.zeros(s.size)

for i, si in enumerate(s):
    c, b = roots_legendre(si)
    c = 0.5*(c+1)
    b = 0.5*b
    res = np.sum(b * f(c))
    f_pert = f(c) + eps*(2*np.random.rand(c.size)-1)*f(c)
    res_pert = np.sum(b * f_pert)
    err_pert[i] = np.abs(res - res_pert)
    
fig = go.Figure()
fig.add_trace(go.Scatter(x=s, y=err_pert, mode='lines+markers'))
fig.update_xaxes(title="Number of quadrature points")
fig.update_yaxes(type="log", exponentformat = 'e', title="|Res. - Perturbed res.|")
fig.show()

The conditioning is excellent here and the perturbation is not amplified, whatever the order of the method used. 

**Result for the function $g(x) = \sqrt(x) \log(x)$**

In [ ]:
def g(x):
    return  np.sqrt(x)*np.log(x)

In [ ]:
res_exa_g = -4./9.

s = np.arange(2, 41, 1)

err = np.zeros(s.size)

for i, si in enumerate(s):
    c, b = roots_legendre(si)
    c = 0.5*(c+1)
    b = 0.5*b
    res = np.sum(b * g(c))
    err[i] = np.abs(res - res_exa_g)
    if (err[i] == 0): err[i]=1e-16

fig = go.Figure()
fig.add_trace(go.Scatter(x=s, y=err, mode='lines+markers'))
fig.update_xaxes(title="Number of quadrature points")
fig.update_yaxes(type="log", exponentformat = 'e', title="Error")
fig.show()